# Validation against Braginsky et al. (2019)

Their published predictor values come from `uni_joined.RData` in the reference repository. The norms are the part both pipelines share: same source files, same replacement tables. Anything that differs here is a processing difference, not a data difference.

In [ ]:
import os
import sys

# notebooks live one level down, src is imported from the project root
if "src" not in os.listdir():
    os.chdir("..")
sys.path.insert(0, os.getcwd())

In [ ]:
import warnings
import matplotlib.pyplot as plt
import pandas as pd
import rdata
from src.common import schema
from src.common.config import raw_dir

In [ ]:
NORMS = ["concreteness", "valence", "arousal", "babiness"]

ours = schema.read("eng")

# their data frame is a tibble, which pyreadr cannot read
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    parsed = rdata.parser.parse_file(raw_dir("braginsky") / "uni_joined.RData")
    theirs = rdata.conversion.convert(parsed)["uni_joined"]

theirs = theirs[theirs["language"] == "English (American)"]
theirs = theirs.drop_duplicates("uni_lemma")[["uni_lemma"] + NORMS]

len(ours), len(theirs)

## Lemma coverage

Wordbank renamed some lemmas between versions, so the two sets do not match
one to one. This has to be settled before comparing values.

In [ ]:
mine = set(ours["uni_lemma"])
hers = set(theirs["uni_lemma"])

print(len(mine & hers), "shared")
print(sorted(mine - hers))

## Norm values

Both pipelines read the same three norm files and the same replacement tables,
so the values should be identical, not merely correlated. Anything that
differs comes from how the tables are applied.

In [ ]:
both = ours.merge(theirs, on="uni_lemma", suffixes=("_ours", "_theirs"))

for norm in NORMS:
    a, b = both[f"{norm}_ours"], both[f"{norm}_theirs"]
    equal = (a - b).abs() < 1e-6
    differ = ~equal & a.notna() & b.notna()
    one_sided = a.isna() != b.isna()
    print(norm, equal.sum(), "equal,", differ.sum(), "differ,", one_sided.sum(), "one-sided")

All four norms match exactly on the shared lemmas, and the gaps sit in the same places on both sides. The norm side of the pipeline is validated: same files, same replacement tables, same result.

The nine lemmas we have and they do not are a Wordbank version difference, not a processing difference. Six are pronouns, which Wordbank 2.0 renamed to glossing codes (1SG, 3SG.POSS). Braginsky’s replacement tables do not know these codes, so src/norms/english.py maps each one to its English words. The other three, blocks, carrot and key, are ordinary words whose lemma names must have changed too, but we did not trace them.

## Age of acquisition

Nothing to validate against here: the 2019 pipeline models proportions directly and never estimates an age of acquisition. The estimates below follow Braginsky et al. (2016) instead.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))

for category, group in ours.groupby("lexical_category"):
    ax.hist(group["aoa"], bins=20, alpha=0.5, label=category)

ax.set_xlabel("age of acquisition (months)")
ax.set_ylabel("lemmas")
ax.legend()
plt.show()

In [ ]:
ours.groupby("lexical_category")["aoa"].describe()

In [ ]:
ours["aoa"].min(), ours["aoa"].max()

The three categories fall in the expected order by median age of acquisition: nouns at 23.8 months, predicates at 25.8, function words at 27.7. Function words are the smallest group at 33 lemmas and are almost hidden behind the other two in the histogram.

The spread is narrower than an unbounded estimator allows. Nothing falls below 15 months or above 37, although -intercept/slope can return any value. The data cover the rise of most curves, so few estimates have to be extrapolated.